# GPT-2 Inference Engine - Google Colab Setup

This notebook sets up the GPT-2 inference engine on Google Colab with T4 GPU acceleration.

**Runtime:** GPU (T4) | **Estimated time:** 15-20 minutes

## Step 1: Environment Setup

Check GPU availability and install build dependencies.

In [1]:
# Check GPU
!nvidia-smi

Thu Apr  2 12:02:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Install build dependencies
!apt-get update && apt-get install -y build-essential cmake git wget curl htop

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,480 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,963 kB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packag

In [3]:
# Check CUDA version
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


## Step 2: Clone and Build GGML with CUDA Support

GGML provides the tensor computation backend with CUDA acceleration.

In [4]:
# Clone GGML (using master branch for compatibility)
%cd /content
!rm -rf ggml && git clone https://github.com/ggerganov/ggml.git --depth 1 ggml

/content
Cloning into 'ggml'...
remote: Enumerating objects: 1948, done.
remote: Counting objects: 100% (1948/1948), done.
remote: Compressing objects: 100% (1498/1498), done.
remote: Total 1948 (delta 507), reused 1509 (delta 435), pack-reused 0 (from 0)
Receiving objects: 100% (1948/1948), 3.01 MiB | 13.23 MiB/s, done.
Resolving deltas: 100% (507/507), done.


In [5]:
# Build GGML with CUDA support
%cd /content/ggml
!rm -rf build && mkdir -p build && cd build && cmake .. -DGGML_CUDA=ON -DCMAKE_BUILD_TYPE=Release && make -j$(nproc)

/content/ggml
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- Found OpenMP_C: -fopenmp (found version "4.5")
-- Found OpenMP_CXX: -fopenmp (f

In [6]:
# Verify GGML build with CUDA
!cd /content/ggml/build && ls -la src/

total 1860
drwxr-xr-x 5 root root    4096 Apr  2 12:44 .
drwxr-xr-x 7 root root    4096 Apr  2 12:02 ..
drwxr-xr-x 5 root root    4096 Apr  2 12:02 CMakeFiles
-rw-r--r-- 1 root root    3104 Apr  2 12:02 cmake_install.cmake
drwxr-xr-x 3 root root    4096 Apr  2 12:02 ggml-cpu
drwxr-xr-x 3 root root    4096 Apr  2 12:44 ggml-cuda
lrwxrwxrwx 1 root root      17 Apr  2 12:03 libggml-base.so -> libggml-base.so.0
lrwxrwxrwx 1 root root      22 Apr  2 12:03 libggml-base.so.0 -> libggml-base.so.0.9.11
-rwxr-xr-x 1 root root  746896 Apr  2 12:03 libggml-base.so.0.9.11
lrwxrwxrwx 1 root root      16 Apr  2 12:04 libggml-cpu.so -> libggml-cpu.so.0
lrwxrwxrwx 1 root root      21 Apr  2 12:04 libggml-cpu.so.0 -> libggml-cpu.so.0.9.11
-rwxr-xr-x 1 root root 1036712 Apr  2 12:04 libggml-cpu.so.0.9.11
lrwxrwxrwx 1 root root      12 Apr  2 12:44 libggml.so -> libggml.so.0
lrwxrwxrwx 1 root root      17 Apr  2 12:44 libggml.so.0 -> libggml.so.0.9.11
-rwxr-xr-x 1 root root   55184 Apr  2 12:44 libggml.so

## Step 3: Clone Inference Engine

Clone your inference engine source files to the Colab environment.

In [23]:
# Clone the inference engine repo (includes CMakeLists.txt)
%cd /content
!rm -rf inference-engine && git clone https://github.com/nisbenz/inference-engine.git inference-engine

# Verify CMakeLists.txt exists
!ls -la /content/inference-engine/

/content
Cloning into 'inference-engine'...
remote: Enumerating objects: 675, done.
remote: Counting objects: 100% (231/231), done.
remote: Compressing objects: 100% (153/153), done.
remote: Total 675 (delta 156), reused 144 (delta 76), pack-reused 444 (from 1)
Receiving objects: 100% (675/675), 279.69 KiB | 11.65 MiB/s, done.
Resolving deltas: 100% (468/468), done.
total 176
drwxr-xr-x 5 root root   4096 Apr  2 13:30 .
drwxr-xr-x 1 root root   4096 Apr  2 13:30 ..
-rw-r--r-- 1 root root   3408 Apr  2 13:30 CMakeLists.txt
-rw-r--r-- 1 root root 141552 Apr  2 13:30 colab_gpt2_inference.ipynb
drwxr-xr-x 8 root root   4096 Apr  2 13:30 .git
-rw-r--r-- 1 root root     36 Apr  2 13:30 .gitignore
-rw-r--r-- 1 root root   1064 Apr  2 13:30 LICENSE
-rw-r--r-- 1 root root   2763 Apr  2 13:30 readme.md
drwxr-xr-x 2 root root   4096 Apr  2 13:30 src
drwxr-xr-x 2 root root   4096 Apr  2 13:30 tests


In [8]:
# Upload your source files
# Use the Files panel to upload src/model.cpp, src/model.hpp, src/layers.cpp,
# src/layers.hpp, src/tokenizer.cpp, src/tokenizer.hpp, src/kv_cache.cpp,
# src/kv_cache.hpp, src/main.cpp, src/gguf_loader.cpp, src/gguf_loader.h

# Or run this cell after uploading via the Files panel
import os

src_dir = '/content/inference-engine/src'
print(f'Source files in {src_dir}:')
for f in os.listdir(src_dir):
    print(f'  {f}')

Source files in /content/inference-engine/src:
  tokenizer.cpp
  model.hpp
  gguf_loader.cpp
  model.cpp
  layers.cpp
  kv_cache.cpp
  main.cpp
  tokenizer.hpp
  gguf_loader.h
  kv_cache.hpp
  layers.hpp


## Step 4: Download GPT-2 Model (GGUF Format)

Download pre-converted GPT-2 GGUF model from HuggingFace.

**Model:** Mungert/gpt2-GGUF (F16 precision for best quality on T4 GPU)
**Config:** 12 layers, 768 hidden, 12 heads, 124M parameters

In [9]:
# Install HuggingFace Hub for downloading models
!pip install huggingface_hub -q

In [10]:
# List available GGUF files in the new repository
from huggingface_hub import hf_hub_download
from huggingface_hub import list_repo_files

print("Available GGUF files in Mungert/gpt2-GGUF:")
files = list_repo_files("Mungert/gpt2-GGUF")
for f in files:
    if f.endswith('.gguf'):
        print(f"  {f}")

Available GGUF files in Mungert/gpt2-GGUF:
  gpt2-bf16.gguf
  gpt2-bf16_q8_0.gguf
  gpt2-f16_q8_0.gguf
  gpt2-iq1_m.gguf
  gpt2-iq1_s.gguf
  gpt2-iq2_m.gguf
  gpt2-iq2_s.gguf
  gpt2-iq2_xs.gguf
  gpt2-iq2_xxs.gguf
  gpt2-iq3_m.gguf
  gpt2-iq3_s.gguf
  gpt2-iq3_xs.gguf
  gpt2-iq3_xxs.gguf
  gpt2-iq4_nl.gguf
  gpt2-iq4_xs.gguf
  gpt2-q2_k_m.gguf
  gpt2-q2_k_s.gguf
  gpt2-q3_k_m.gguf
  gpt2-q3_k_s.gguf
  gpt2-q4_0.gguf
  gpt2-q4_1.gguf
  gpt2-q4_k_m.gguf
  gpt2-q4_k_s.gguf
  gpt2-q5_0.gguf
  gpt2-q5_1.gguf
  gpt2-q5_k_m.gguf
  gpt2-q5_k_s.gguf
  gpt2-q6_k_m.gguf
  gpt2-q8_0.gguf
  gpt2-tq1_0.gguf
  gpt2-tq2_0.gguf


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [11]:
# Create model directory
!mkdir -p /content/gpt2-model

# Download GPT-2 F16 model (226 MB) - full precision for best quality
# For T4 GPU with 16GB VRAM, F16 fits easily and gives best results
print("Downloading GPT-2 F16 model...")
model_path = hf_hub_download(
    repo_id="Mungert/gpt2-GGUF",
    filename="gpt2-bf16.gguf",
    repo_type="model",
    local_dir="/content/gpt2-model"
)
print(f"Model downloaded to: {model_path}")

gpt2-bf16.gguf:   0%|          | 0.00/252M [00:00<?, ?B/s]

Model downloaded to: /content/gpt2-model/gpt2-bf16.gguf


In [12]:
# Download GPT-2 tokenizer files from HuggingFace
from huggingface_hub import hf_hub_download
print("Downloading tokenizer files...")
hf_hub_download(repo_id="openai-community/gpt2", filename="vocab.json", local_dir="/content/gpt2-model")
hf_hub_download(repo_id="openai-community/gpt2", filename="merges.txt", local_dir="/content/gpt2-model")

# Verify files are not empty
!ls -lh /content/gpt2-model/


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

total 243M
-rw-r--r-- 1 root root  241M Apr  2 12:45 gpt2-bf16.gguf
-rw-r--r-- 1 root root  446K Apr  2 12:45 merges.txt
-rw-r--r-- 1 root root 1018K Apr  2 12:45 vocab.json


In [24]:
# Update model path in main.cpp
# This ensures the correct model file is loaded
import re

main_cpp_path = '/content/inference-engine/src/main.cpp'
with open(main_cpp_path, 'r') as f:
    content = f.read()

# Update weights path to use F16 model
content = re.sub(
    r'std::string weights_path = "[^"]*"',
    'std::string weights_path = "/content/gpt2-model/gpt2-bf16.gguf"',
    content
)

# Update tokenizer paths
content = re.sub(
    r'std::string vocab_path = "[^"]*"',
    'std::string vocab_path = "/content/gpt2-model/vocab.json"',
    content
)
content = re.sub(
    r'std::string merges_path = "[^"]*"',
    'std::string merges_path = "/content/gpt2-model/merges.txt"',
    content
)

with open(main_cpp_path, 'w') as f:
    f.write(content)

print("Updated main.cpp with correct file paths:")
!grep -n "weights_path\|vocab_path\|merges_path" /content/inference-engine/src/main.cpp

Updated main.cpp with correct file paths:
49:    std::string weights_path = "/content/gpt2-model/gpt2-bf16.gguf";
50:    std::cout << "Loading weights from: " << weights_path << std::endl;
52:    if (!model.load_weights(weights_path)) {
57:    std::string vocab_path = "/content/gpt2-model/vocab.json";
58:    std::string merges_path = "/content/gpt2-model/merges.txt";
59:    std::cout << "Loading tokenizer from: " << vocab_path << " and " << merges_path << std::endl;
60:    if (!model.load_tokenizer(vocab_path, merges_path)) {


## Step 5: Build Custom Inference Engine

Build your own inference engine with the updated source code.

In [25]:
# Build the inference engine
# Note: GGML_DIR must point to the ggml source directory so cmake can find ggml/ggml.h
!cd /content/inference-engine && mkdir -p build && cd build && cmake .. -DCMAKE_CUDA_PATH=/usr/local/cuda -DCMAKE_CUDA_ARCHITECTURES=75 -DGGML_DIR=/content/ggml && make -j$(nproc)

-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- The CUDA compiler identification is NVIDIA 12.8.93 with host compiler GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Detecting CUDA compiler ABI info
-- Detecting CUDA compiler ABI info - done
-- Check for working CUDA compiler: /usr/local/cuda/bin/nvcc - skipped
-- Detecting CUDA compile features
-- Detecting CUDA compile features - done
CMake Warning (dev) at CMakeLists.txt:9 (find_package):
  Policy CMP0146 is not set: The FindCUDA module is removed.  Run "cmake
  --help-policy CMP0146" for policy details.  Use the c

## Step 6: Run Inference

Execute your inference engine with GPU acceleration.

**Usage:** `./gpt2 "prompt" [max_tokens] [temperature] [top_k]`

In [27]:
# Run with your inference engine
!cd /content/inference-engine/build && ./gpt2 "the american flag is red and white and" 50 0.3 40

=== GPT-2 Large Inference ===
Prompt: "the american flag is red and white and"
Max tokens: 50
Temperature: 0.3
Top-k: 40
ggml_cuda_init: found 1 CUDA devices (Total VRAM: 14912 MiB):
  Device 0: Tesla T4, compute capability 7.5, VMM: yes, VRAM: 14912 MiB
GPT-2 Large model initialized
  Layers: 12
  Hidden: 768
  Heads: 12
  FFN: 3072
  Vocab: 50257
Loading weights from: /content/gpt2-model/gpt2-bf16.gguf
Loading GGUF model from: /content/gpt2-model/gpt2-bf16.gguf
Model architecture: gpt2
Config: ctx=1024 embd=768 head=12 layer=12 ffn=3072

Sample tensor names:
  blk.0.attn_qkv.bias
  blk.0.attn_qkv.weight
  blk.0.attn_output.bias
  blk.0.attn_output.weight
  blk.0.attn_norm.bias
  ... (148 total tensors)

Loading tensors...

Loaded 148 tensors, 0 failed/skipped
GGUF model loaded successfully
Loading tokenizer from: /content/gpt2-model/vocab.json and /content/gpt2-model/merges.txt
Loaded 50257 vocab entries
Loaded 49996 BPE merges
Tokenizing prompt...
Prompt tokens: 9 tokens

Generating

In [28]:
!cd /content/inference-engine/build/bin/ && ./run_tests

       GPT-2 Inference Engine Unit Tests         

Running GGUF Loader Tests

=== test_type_sizes ===
  All GGUF type IDs verified correctly

=== test_bf16_conversion ===
  BF16 conversion verified

=== test_fp16_conversion ===
  FP16 conversion verified

=== test_tensor_info_structure ===
  Tensor info structure verified

=== test_metadata_value_types ===
  Metadata value types verified

=== test_q80_alt_dequantization ===
  Q8_0_ALT dequantization verified (scale=1)

=== test_layer_idx_extraction ===
  Layer index extraction verified

=== test_transpose_detection ===
  Transpose detection logic verified

=== test_transpose_operation ===
  Transpose operation verified

=== test_q80_alt_actual_file ===
  No GGUF path provided, skipping

=== test_actual_tensor_data_size ===
  No GGUF path provided, skipping

All GGUF Loader Tests PASSED

Running Model Loading Tests

=== test_model_config ===
  Model config verified: 12 layers, 768 hidden, 12 heads, 3072 FFN

=== test_tensor_shapes ===
 